# MLB 피치클락 도입 전후 투수 부상률 분석 및 인과 식별 진단

이 노트북은 2022년과 2023년 투수 자료를 이용해 부상률의 **전후 차이**를 계산하고, 동일 투수 패널과 여러 분석 표본에서 결과가 얼마나 달라지는지 확인합니다.

핵심 목적은 숫자 하나를 인과효과로 확정하는 것이 아니라 다음 질문을 구분하는 것입니다.

1. 업로드된 자료에서 2022년과 2023년 부상률은 얼마나 달랐는가?
2. 동일 투수만 비교하면 결과가 유지되는가?
3. 기존 성향점수 분석은 인과효과를 식별할 조건을 충족했는가?

> **해석 범위:** 2023년에는 모든 MLB 투수가 피치클락 환경에 노출됐고 동시기 비노출 비교군이 없습니다. 따라서 이 자료만으로 피치클락의 인과효과를 식별할 수 없습니다. 결과는 관찰자료의 전후 연관성과 분석 설계 진단으로 해석합니다.

In [1]:
from __future__ import annotations

import csv
import math
import re
import unicodedata
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.proportion import confint_proportions_2indep, proportion_confint

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "raw"
OUTPUT_DIR = PROJECT_DIR / "outputs"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(exist_ok=True)
FIGURE_DIR.mkdir(exist_ok=True)

RANDOM_SEED = 42
BOOTSTRAP_REPETITIONS = 20_000

print(f"Project directory: {PROJECT_DIR}")
print(f"Raw data directory: {DATA_DIR}")

Project directory: /mnt/data/MLB_pitch_clock_revised
Raw data directory: /mnt/data/MLB_pitch_clock_revised/data/raw


## 1. 데이터 로드와 공통 함수

In [2]:
def read_csv_rows(path: Path) -> list[dict[str, str]]:
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        return list(csv.DictReader(file))


def to_float(value: object) -> float:
    try:
        text = str(value).strip()
        return float(text) if text else math.nan
    except (TypeError, ValueError):
        return math.nan


def to_int(value: object) -> int | None:
    number = to_float(value)
    return None if math.isnan(number) else int(number)


def normalize_name(value: str) -> str:
    text = unicodedata.normalize("NFKC", str(value)).replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text.strip().lower())
    return text


def write_dict_rows(path: Path, rows: list[dict], fieldnames: list[str] | None = None) -> None:
    if not rows:
        raise ValueError(f"No rows to write: {path}")
    columns = fieldnames or list(rows[0].keys())
    with path.open("w", encoding="utf-8-sig", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)


def wilson_interval(events: int, total: int) -> tuple[float, float]:
    low, high = proportion_confint(events, total, method="wilson")
    return float(low), float(high)


def primary_velocity(row: dict[str, str]) -> float:
    # Reproduces the original notebook definition: four-seam -> sinker -> cutter.
    for column in ("ff_avg_speed", "si_avg_speed", "fc_avg_speed"):
        value = to_float(row.get(column, ""))
        if not math.isnan(value):
            return value
    return math.nan


def standardized_mean_difference(treated: np.ndarray, control: np.ndarray) -> float:
    pooled = math.sqrt((np.var(treated, ddof=1) + np.var(control, ddof=1)) / 2)
    return float((np.mean(treated) - np.mean(control)) / pooled)


raw_2022_tempo = read_csv_rows(DATA_DIR / "2022_pitch_tempo.csv")
raw_2023_tempo = read_csv_rows(DATA_DIR / "2023_pitch_tempo.csv")
raw_2022_velocity = read_csv_rows(DATA_DIR / "2022_pitch_velocity.csv")
raw_2023_velocity = read_csv_rows(DATA_DIR / "2023_pitch_velocity.csv")
merged_2022 = read_csv_rows(DATA_DIR / "2022_merged_data.csv")
merged_2023 = read_csv_rows(DATA_DIR / "2023_merged_data.csv")

print("Loaded rows")
print(f"- 2022 tempo: {len(raw_2022_tempo)}")
print(f"- 2023 tempo: {len(raw_2023_tempo)}")
print(f"- 2022 velocity: {len(raw_2022_velocity)}")
print(f"- 2023 velocity: {len(raw_2023_velocity)}")
print(f"- 2022 tempo-velocity merged: {len(merged_2022)}")
print(f"- 2023 tempo-velocity merged: {len(merged_2023)}")

Loaded rows
- 2022 tempo: 391
- 2023 tempo: 251
- 2022 velocity: 352
- 2023 velocity: 361
- 2022 tempo-velocity merged: 348
- 2023 tempo-velocity merged: 212


## 2. 선수 단위 정제와 중복 처리

In [3]:
def prepare_2022_tempo(rows: list[dict[str, str]]) -> list[dict]:
    cleaned = []
    for row in rows:
        # 2022 file stores last name and first name in separate columns.
        first_name = row.get("", row.get("Unnamed: 1", "")).strip()
        last_name = row.get("entity_name", "").strip()
        player = f"{first_name} {last_name}".strip()
        cleaned.append({
            "year": 2022,
            "player": player,
            "player_key": normalize_name(player),
            "tempo_seconds_empty": to_float(row.get("median_seconds_empty")),
            "pitches_empty": to_int(row.get("total_pitches")),
            "injury": to_int(row.get("arm_injury_2022")),
            "source_records": 1,
        })
    return cleaned


def prepare_2023_tempo(rows: list[dict[str, str]]) -> list[dict]:
    # One player (Andrew Heaney) appears twice. Aggregate team-split rows to one player-season.
    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[normalize_name(row.get("Player", ""))].append(row)

    cleaned = []
    for player_key, group in grouped.items():
        pitches = np.array([to_float(row.get("total_pitches_empty")) for row in group], dtype=float)
        tempos = np.array([to_float(row.get("Tempo_23")) for row in group], dtype=float)
        valid = np.isfinite(pitches) & np.isfinite(tempos)
        if not valid.any() or pitches[valid].sum() <= 0:
            raise ValueError(f"Cannot aggregate tempo for {group[0].get('Player')}")
        injury_values = {to_int(row.get("Inj_23")) for row in group}
        if len(injury_values) != 1:
            raise ValueError(f"Conflicting injury labels for {group[0].get('Player')}")
        cleaned.append({
            "year": 2023,
            "player": group[0].get("Player", "").strip(),
            "player_key": player_key,
            "tempo_seconds_empty": float(np.average(tempos[valid], weights=pitches[valid])),
            "pitches_empty": int(pitches[valid].sum()),
            "injury": injury_values.pop(),
            "source_records": len(group),
        })
    return cleaned


def prepare_merged_2022(rows: list[dict[str, str]]) -> list[dict]:
    output = []
    for row in rows:
        output.append({
            "year": 2022,
            "player": row.get("Player", "").strip(),
            "player_key": normalize_name(row.get("Player", "")),
            "player_id": row.get("pitcher", "").strip(),
            "tempo_seconds_empty": to_float(row.get("median_seconds_empty")),
            "pitches_empty": to_int(row.get("total_pitches")),
            "primary_velocity_mph": primary_velocity(row),
            "injury": to_int(row.get("arm_injury_2022")),
        })
    return output


def prepare_merged_2023(rows: list[dict[str, str]]) -> list[dict]:
    grouped: dict[str, list[dict[str, str]]] = defaultdict(list)
    for row in rows:
        grouped[row.get("pitcher", "").strip()].append(row)

    output = []
    for player_id, group in grouped.items():
        pitches = np.array([to_float(row.get("total_pitches_empty")) for row in group], dtype=float)
        tempos = np.array([to_float(row.get("Tempo_23")) for row in group], dtype=float)
        valid = np.isfinite(pitches) & np.isfinite(tempos)
        injuries = {to_int(row.get("Inj_23")) for row in group}
        if len(injuries) != 1:
            raise ValueError(f"Conflicting injury labels for player ID {player_id}")
        velocities = np.array([primary_velocity(row) for row in group], dtype=float)
        output.append({
            "year": 2023,
            "player": group[0].get("Player", "").strip(),
            "player_key": normalize_name(group[0].get("Player", "")),
            "player_id": player_id,
            "tempo_seconds_empty": float(np.average(tempos[valid], weights=pitches[valid])),
            "pitches_empty": int(pitches[valid].sum()),
            "primary_velocity_mph": float(np.nanmean(velocities)),
            "injury": injuries.pop(),
        })
    return output


tempo_2022 = prepare_2022_tempo(raw_2022_tempo)
tempo_2023 = prepare_2023_tempo(raw_2023_tempo)
complete_2022 = prepare_merged_2022(merged_2022)
complete_2023 = prepare_merged_2023(merged_2023)

assert len(tempo_2022) == 391
assert len(tempo_2023) == 250
assert sum(row["source_records"] - 1 for row in tempo_2023) == 1
assert len(complete_2022) == 348
assert len(complete_2023) == 211
assert len({row["player_key"] for row in tempo_2022}) == len(tempo_2022)
assert len({row["player_key"] for row in tempo_2023}) == len(tempo_2023)
assert all(row["injury"] in (0, 1) for row in tempo_2022 + tempo_2023)

complete_by_name_2022 = {row["player_key"]: row for row in complete_2022}
complete_by_name_2023 = {row["player_key"]: row for row in complete_2023}

analysis_sample = []
for row in tempo_2022 + tempo_2023:
    complete_map = complete_by_name_2022 if row["year"] == 2022 else complete_by_name_2023
    complete = complete_map.get(row["player_key"])
    analysis_sample.append({
        **row,
        "in_velocity_merge": int(complete is not None),
        "player_id": complete.get("player_id", "") if complete else "",
        "primary_velocity_mph": complete.get("primary_velocity_mph", "") if complete else "",
    })

write_dict_rows(OUTPUT_DIR / "analysis_sample.csv", analysis_sample)

print("Unique player-season records")
print(f"- 2022: {len(tempo_2022)}")
print(f"- 2023: {len(tempo_2023)} (1 duplicate row aggregated)")
print(f"- Complete tempo-velocity sample: {len(complete_2022)} in 2022, {len(complete_2023)} in 2023")

Unique player-season records
- 2022: 391
- 2023: 250 (1 duplicate row aggregated)
- Complete tempo-velocity sample: 348 in 2022, 211 in 2023


## 3. 데이터 품질과 표본 선택 진단

In [4]:
def injury_summary(rows: list[dict]) -> tuple[int, int, float]:
    total = len(rows)
    events = sum(int(row["injury"]) for row in rows)
    return total, events, events / total


def retention_rows(year: int, tempo_rows: list[dict], complete_keys: set[str]) -> list[dict]:
    output = []
    for injury in (0, 1):
        group = [row for row in tempo_rows if row["injury"] == injury]
        retained = sum(row["player_key"] in complete_keys for row in group)
        output.append({
            "year": year,
            "injury": injury,
            "tempo_unique_players": len(group),
            "retained_in_velocity_merge": retained,
            "retention_rate": retained / len(group),
        })
    return output

keys_2022_complete = {row["player_key"] for row in complete_2022}
keys_2023_complete = {row["player_key"] for row in complete_2023}
common_name_keys = sorted({row["player_key"] for row in tempo_2022} & {row["player_key"] for row in tempo_2023})
common_id_keys = sorted({row["player_id"] for row in complete_2022} & {row["player_id"] for row in complete_2023})

velocity_2022_by_id = {row.get("pitcher", "").strip(): row for row in raw_2022_velocity}
velocity_2023_by_id = {row.get("pitcher", "").strip(): row for row in raw_2023_velocity}


def velocity_crosscheck(merged_rows: list[dict[str, str]], velocity_map: dict[str, dict[str, str]]) -> tuple[int, float]:
    matched = 0
    differences = []
    for row in merged_rows:
        player_id = row.get("pitcher", "").strip()
        raw_velocity_row = velocity_map.get(player_id)
        if raw_velocity_row is None:
            continue
        matched += 1
        merged_velocity = primary_velocity(row)
        raw_velocity = primary_velocity(raw_velocity_row)
        if math.isfinite(merged_velocity) and math.isfinite(raw_velocity):
            differences.append(abs(merged_velocity - raw_velocity))
    max_difference = max(differences) if differences else math.nan
    return matched, max_difference


velocity_matches_2022, velocity_max_diff_2022 = velocity_crosscheck(merged_2022, velocity_2022_by_id)
velocity_matches_2023, velocity_max_diff_2023 = velocity_crosscheck(merged_2023, velocity_2023_by_id)
assert velocity_matches_2022 == len(merged_2022)
assert velocity_matches_2023 == len(merged_2023)
assert velocity_max_diff_2022 == 0
assert velocity_max_diff_2023 == 0

data_audit = [
    {"metric": "2022 raw tempo rows", "value": len(raw_2022_tempo), "note": "No duplicate player names"},
    {"metric": "2022 unique players", "value": len(tempo_2022), "note": "Primary 2022 sample"},
    {"metric": "2022 raw velocity rows", "value": len(raw_2022_velocity), "note": "Unique Statcast pitcher IDs"},
    {"metric": "2022 tempo-velocity merged players", "value": len(complete_2022), "note": f"{len(complete_2022)/len(tempo_2022):.1%} retained"},
    {"metric": "2022 merged rows cross-checked to velocity export", "value": velocity_matches_2022, "note": f"Maximum primary-velocity difference {velocity_max_diff_2022:.1f} mph"},
    {"metric": "2023 raw tempo rows", "value": len(raw_2023_tempo), "note": "Includes two Andrew Heaney rows"},
    {"metric": "2023 unique players", "value": len(tempo_2023), "note": "Duplicate aggregated by pitch-count weighting"},
    {"metric": "2023 raw velocity rows", "value": len(raw_2023_velocity), "note": "Unique Statcast pitcher IDs"},
    {"metric": "2023 tempo-velocity merged players", "value": len(complete_2023), "note": f"{len(complete_2023)/len(tempo_2023):.1%} retained"},
    {"metric": "2023 merged rows cross-checked to velocity export", "value": velocity_matches_2023, "note": f"Maximum primary-velocity difference {velocity_max_diff_2023:.1f} mph"},
    {"metric": "Matched players by normalized name", "value": len(common_name_keys), "note": "Used for paired primary sensitivity"},
    {"metric": "Matched players by Statcast ID", "value": len(common_id_keys), "note": "Complete-case paired sensitivity"},
]
retention_by_injury = (
    retention_rows(2022, tempo_2022, keys_2022_complete)
    + retention_rows(2023, tempo_2023, keys_2023_complete)
)

write_dict_rows(OUTPUT_DIR / "data_audit.csv", data_audit)
write_dict_rows(OUTPUT_DIR / "merge_retention_by_injury.csv", retention_by_injury)

for row in data_audit:
    print(f"{row['metric']}: {row['value']} ({row['note']})")
print("\nRetention by injury status")
for row in retention_by_injury:
    print(
        f"{row['year']} injury={row['injury']}: "
        f"{row['retained_in_velocity_merge']}/{row['tempo_unique_players']} "
        f"({row['retention_rate']:.1%})"
    )

2022 raw tempo rows: 391 (No duplicate player names)
2022 unique players: 391 (Primary 2022 sample)
2022 raw velocity rows: 352 (Unique Statcast pitcher IDs)
2022 tempo-velocity merged players: 348 (89.0% retained)
2022 merged rows cross-checked to velocity export: 348 (Maximum primary-velocity difference 0.0 mph)
2023 raw tempo rows: 251 (Includes two Andrew Heaney rows)
2023 unique players: 250 (Duplicate aggregated by pitch-count weighting)
2023 raw velocity rows: 361 (Unique Statcast pitcher IDs)
2023 tempo-velocity merged players: 211 (84.4% retained)
2023 merged rows cross-checked to velocity export: 212 (Maximum primary-velocity difference 0.0 mph)
Matched players by normalized name: 188 (Used for paired primary sensitivity)
Matched players by Statcast ID: 155 (Complete-case paired sensitivity)

Retention by injury status
2022 injury=0: 265/300 (88.3%)
2022 injury=1: 83/91 (91.2%)
2023 injury=0: 158/181 (87.3%)
2023 injury=1: 53/69 (76.8%)


## 4. 전후 부상률 차이

주 분석은 중복을 제거한 선수-시즌 단위 표본을 사용합니다. 연도별 선수 구성이 다르므로 이 추정치는 **독립 표본의 전후 차이**입니다.

- 위험도 차이: 2023 부상률 − 2022 부상률
- 95% 신뢰구간: Newcombe 방식
- 위험비 신뢰구간: 로그 방식
- 검정: Fisher의 정확검정

In [5]:
def independent_effect(label: str, rows_2022: list[dict], rows_2023: list[dict]) -> dict:
    n_2022, e_2022, risk_2022 = injury_summary(rows_2022)
    n_2023, e_2023, risk_2023 = injury_summary(rows_2023)
    rd = risk_2023 - risk_2022
    rd_low, rd_high = confint_proportions_2indep(
        e_2023, n_2023, e_2022, n_2022,
        compare="diff", method="newcomb"
    )
    rr = risk_2023 / risk_2022
    rr_low, rr_high = confint_proportions_2indep(
        e_2023, n_2023, e_2022, n_2022,
        compare="ratio", method="log-adjusted"
    )
    p_value = stats.fisher_exact(
        [[e_2023, n_2023 - e_2023], [e_2022, n_2022 - e_2022]]
    ).pvalue
    return {
        "analysis": label,
        "design": "Independent before-after comparison",
        "n_2022": n_2022,
        "injured_2022": e_2022,
        "risk_2022": risk_2022,
        "n_2023": n_2023,
        "injured_2023": e_2023,
        "risk_2023": risk_2023,
        "risk_difference": rd,
        "risk_difference_ci_low": float(rd_low),
        "risk_difference_ci_high": float(rd_high),
        "risk_ratio": rr,
        "risk_ratio_ci_low": float(rr_low),
        "risk_ratio_ci_high": float(rr_high),
        "p_value": float(p_value),
        "test": "Fisher exact",
    }

primary_effect = independent_effect("All unique tempo records", tempo_2022, tempo_2023)
complete_case_effect = independent_effect("Tempo-velocity complete cases", complete_2022, complete_2023)

for result in (primary_effect, complete_case_effect):
    print(result["analysis"])
    print(
        f"  2022: {result['injured_2022']}/{result['n_2022']} = {result['risk_2022']:.1%}\n"
        f"  2023: {result['injured_2023']}/{result['n_2023']} = {result['risk_2023']:.1%}\n"
        f"  Risk difference: {result['risk_difference']*100:+.2f} percentage points "
        f"(95% CI {result['risk_difference_ci_low']*100:+.2f} to {result['risk_difference_ci_high']*100:+.2f})\n"
        f"  Risk ratio: {result['risk_ratio']:.2f} "
        f"(95% CI {result['risk_ratio_ci_low']:.2f} to {result['risk_ratio_ci_high']:.2f})\n"
        f"  p-value: {result['p_value']:.3f}"
    )

All unique tempo records
  2022: 91/391 = 23.3%
  2023: 69/250 = 27.6%
  Risk difference: +4.33 percentage points (95% CI -2.49 to +11.36)
  Risk ratio: 1.19 (95% CI 0.91 to 1.55)
  p-value: 0.225
Tempo-velocity complete cases
  2022: 83/348 = 23.9%
  2023: 53/211 = 25.1%
  Risk difference: +1.27 percentage points (95% CI -5.90 to +8.79)
  Risk ratio: 1.05 (95% CI 0.78 to 1.42)
  p-value: 0.761


## 5. 동일 투수 전후 비교

선수 구성 변화를 줄이기 위해 두 시즌에 모두 나타난 투수만 비교합니다. 동일 선수의 두 이진 결과를 비교하므로 McNemar의 정확검정을 사용하고, 위험도 차이의 신뢰구간은 선수 단위 부트스트랩으로 계산합니다.

In [6]:
def paired_effect(label: str, pairs: list[dict], key_2022: str, key_2023: str) -> dict:
    outcomes = np.array([[row[key_2022], row[key_2023]] for row in pairs], dtype=int)
    n = len(outcomes)
    risk_2022 = float(outcomes[:, 0].mean())
    risk_2023 = float(outcomes[:, 1].mean())
    differences = outcomes[:, 1] - outcomes[:, 0]

    rng = np.random.default_rng(RANDOM_SEED)
    indices = rng.integers(0, n, size=(BOOTSTRAP_REPETITIONS, n))
    bootstrapped = differences[indices].mean(axis=1)
    rd_low, rd_high = np.quantile(bootstrapped, [0.025, 0.975])

    transition = np.zeros((2, 2), dtype=int)
    for before, after in outcomes:
        transition[before, after] += 1
    test = mcnemar(transition, exact=True)

    return {
        "analysis": label,
        "design": "Paired before-after comparison",
        "n_2022": n,
        "injured_2022": int(outcomes[:, 0].sum()),
        "risk_2022": risk_2022,
        "n_2023": n,
        "injured_2023": int(outcomes[:, 1].sum()),
        "risk_2023": risk_2023,
        "risk_difference": risk_2023 - risk_2022,
        "risk_difference_ci_low": float(rd_low),
        "risk_difference_ci_high": float(rd_high),
        "risk_ratio": "",
        "risk_ratio_ci_low": "",
        "risk_ratio_ci_high": "",
        "p_value": float(test.pvalue),
        "test": "Exact McNemar",
        "transition_00": int(transition[0, 0]),
        "transition_01": int(transition[0, 1]),
        "transition_10": int(transition[1, 0]),
        "transition_11": int(transition[1, 1]),
    }

map_tempo_2022 = {row["player_key"]: row for row in tempo_2022}
map_tempo_2023 = {row["player_key"]: row for row in tempo_2023}
matched_name_panel = []
for player_key in common_name_keys:
    before = map_tempo_2022[player_key]
    after = map_tempo_2023[player_key]
    matched_name_panel.append({
        "player_key": player_key,
        "player": before["player"],
        "injury_2022": before["injury"],
        "injury_2023": after["injury"],
        "tempo_2022": before["tempo_seconds_empty"],
        "tempo_2023": after["tempo_seconds_empty"],
        "pitches_2022": before["pitches_empty"],
        "pitches_2023": after["pitches_empty"],
    })

map_complete_2022_id = {row["player_id"]: row for row in complete_2022}
map_complete_2023_id = {row["player_id"]: row for row in complete_2023}
matched_id_panel = []
for player_id in common_id_keys:
    before = map_complete_2022_id[player_id]
    after = map_complete_2023_id[player_id]
    matched_id_panel.append({
        "player_id": player_id,
        "player": before["player"],
        "injury_2022": before["injury"],
        "injury_2023": after["injury"],
        "tempo_2022": before["tempo_seconds_empty"],
        "tempo_2023": after["tempo_seconds_empty"],
        "pitches_2022": before["pitches_empty"],
        "pitches_2023": after["pitches_empty"],
        "velocity_2022": before["primary_velocity_mph"],
        "velocity_2023": after["primary_velocity_mph"],
    })

name_matched_effect = paired_effect(
    "Same pitchers matched by normalized name",
    matched_name_panel,
    "injury_2022",
    "injury_2023",
)
id_matched_effect = paired_effect(
    "Same pitchers matched by Statcast ID",
    matched_id_panel,
    "injury_2022",
    "injury_2023",
)

write_dict_rows(OUTPUT_DIR / "matched_player_panel.csv", matched_name_panel)
write_dict_rows(OUTPUT_DIR / "matched_player_id_panel.csv", matched_id_panel)

for result in (name_matched_effect, id_matched_effect):
    print(result["analysis"])
    print(
        f"  n={result['n_2022']}\n"
        f"  2022 risk: {result['risk_2022']:.1%}\n"
        f"  2023 risk: {result['risk_2023']:.1%}\n"
        f"  Risk difference: {result['risk_difference']*100:+.2f} percentage points "
        f"(bootstrap 95% CI {result['risk_difference_ci_low']*100:+.2f} to {result['risk_difference_ci_high']*100:+.2f})\n"
        f"  Exact McNemar p-value: {result['p_value']:.3f}"
    )

Same pitchers matched by normalized name
  n=188
  2022 risk: 27.7%
  2023 risk: 29.3%
  Risk difference: +1.60 percentage points (bootstrap 95% CI -6.91 to +10.11)
  Exact McNemar p-value: 0.810
Same pitchers matched by Statcast ID
  n=155
  2022 risk: 27.7%
  2023 risk: 26.5%
  Risk difference: -1.29 percentage points (bootstrap 95% CI -10.97 to +8.39)
  Exact McNemar p-value: 0.894


## 6. 기준 시점 투구량을 이용한 탐색적 민감도 분석

기존 프로젝트는 각 시즌의 투구 수 중앙값으로 선발·불펜을 구분했습니다. 그러나 같은 시즌 투구 수는 피치클락 이후에 측정되고 부상 발생으로 줄어들 수도 있습니다. 여기서는 동일 투수 패널의 **2022년 투구 수**만 이용해 기준 시점 저·고투구량 집단을 나눕니다.

이 결과도 동시기 대조군이 없는 전후 비교이므로 인과적 이질성으로 해석하지 않습니다.

In [7]:
baseline_pitch_median = float(np.median([row["pitches_2022"] for row in matched_name_panel]))
workload_groups = []
for group_name, selector in (
    ("Lower 2022 workload", lambda row: row["pitches_2022"] < baseline_pitch_median),
    ("Higher 2022 workload", lambda row: row["pitches_2022"] >= baseline_pitch_median),
):
    subset = [row for row in matched_name_panel if selector(row)]
    result = paired_effect(group_name, subset, "injury_2022", "injury_2023")
    result["baseline_pitch_cutoff"] = baseline_pitch_median
    workload_groups.append(result)

write_dict_rows(OUTPUT_DIR / "baseline_workload_sensitivity.csv", workload_groups)

print(f"2022 baseline pitch-count median: {baseline_pitch_median:.0f}")
for result in workload_groups:
    print(
        f"{result['analysis']}: n={result['n_2022']}, "
        f"RD={result['risk_difference']*100:+.2f} pp "
        f"(95% CI {result['risk_difference_ci_low']*100:+.2f} to {result['risk_difference_ci_high']*100:+.2f}), "
        f"p={result['p_value']:.3f}"
    )

2022 baseline pitch-count median: 840
Lower 2022 workload: n=94, RD=-2.13 pp (95% CI -13.83 to +9.57), p=0.864
Higher 2022 workload: n=94, RD=+5.32 pp (95% CI -6.38 to +17.02), p=0.500


## 7. 기존 성향점수 설계 재현과 진단

기존 노트북은 `Tempo`, 같은 시즌 `Pitches`, 같은 시즌 `Velocity`로 2023년 관측치일 확률을 예측했습니다. 이 코드는 기존 설정을 재현하되, 그 결과를 인과효과 추정이 아니라 **식별 실패 진단**으로 사용합니다.

- `Tempo`는 피치클락이 변화시키려는 중간 변수입니다.
- 같은 시즌 `Pitches`는 부상 이후 감소할 수 있습니다.
- 같은 시즌 `Velocity`도 정책 이후 측정된 변수입니다.
- 따라서 이 변수들을 처치 이전 교란변수로 간주해 조정하면 안 됩니다.

In [8]:
# Recreate the original 560-row complete-case dataset, including the duplicate 2023 row.
original_ps_records = []
for row in merged_2022:
    original_ps_records.append({
        "year": 2022,
        "treat": 0,
        "injury": to_int(row.get("arm_injury_2022")),
        "tempo": to_float(row.get("median_seconds_empty")),
        "pitches": to_float(row.get("total_pitches")),
        "velocity": primary_velocity(row),
    })
for row in merged_2023:
    original_ps_records.append({
        "year": 2023,
        "treat": 1,
        "injury": to_int(row.get("Inj_23")),
        "tempo": to_float(row.get("Tempo_23")),
        "pitches": to_float(row.get("total_pitches_empty")),
        "velocity": primary_velocity(row),
    })
original_ps_records = [
    row for row in original_ps_records
    if all(math.isfinite(row[column]) for column in ("tempo", "pitches", "velocity"))
]

X = np.array([[row["tempo"], row["pitches"], row["velocity"]] for row in original_ps_records], dtype=float)
y = np.array([row["treat"] for row in original_ps_records], dtype=int)

ps_model = LogisticRegression(solver="lbfgs", max_iter=1000)
ps_model.fit(X, y)
propensity_score = ps_model.predict_proba(X)[:, 1]
auc = float(roc_auc_score(y, propensity_score))

minimum_treated_score = float(propensity_score[y == 1].min())
maximum_control_score = float(propensity_score[y == 0].max())
in_common_support = (
    (propensity_score >= minimum_treated_score)
    & (propensity_score <= maximum_control_score)
)

balance_rows = []
for index, variable in enumerate(("Tempo", "Pitches", "Velocity")):
    before = standardized_mean_difference(X[y == 1, index], X[y == 0, index])
    after = standardized_mean_difference(
        X[in_common_support & (y == 1), index],
        X[in_common_support & (y == 0), index],
    )
    balance_rows.append({
        "variable": variable,
        "smd_before_trimming": before,
        "smd_after_trimming": after,
        "absolute_smd_after": abs(after),
        "passes_0_1_threshold": int(abs(after) < 0.1),
    })

ps_audit = [
    {"metric": "Rows in original complete-case setup", "value": len(original_ps_records), "interpretation": "Matches the prior 560-row analysis"},
    {"metric": "Propensity model AUC", "value": auc, "interpretation": "Near-perfect year classification indicates very limited comparability"},
    {"metric": "Rows retained by min-max overlap rule", "value": int(in_common_support.sum()), "interpretation": f"{in_common_support.mean():.1%} retained"},
    {"metric": "Rows removed by overlap rule", "value": int((~in_common_support).sum()), "interpretation": f"{(~in_common_support).mean():.1%} removed"},
    {"metric": "Treated rows retained", "value": int((in_common_support & (y == 1)).sum()), "interpretation": "2023 observations"},
    {"metric": "Control rows retained", "value": int((in_common_support & (y == 0)).sum()), "interpretation": "2022 observations"},
]

write_dict_rows(OUTPUT_DIR / "propensity_score_audit.csv", ps_audit)
write_dict_rows(OUTPUT_DIR / "covariate_balance.csv", balance_rows)

print(f"Propensity-score AUC: {auc:.3f}")
print(f"Common-support retention: {in_common_support.sum()}/{len(in_common_support)} ({in_common_support.mean():.1%})")
for row in balance_rows:
    print(
        f"{row['variable']}: SMD before={row['smd_before_trimming']:+.3f}, "
        f"after={row['smd_after_trimming']:+.3f}"
    )

Propensity-score AUC: 0.989
Common-support retention: 230/560 (41.1%)
Tempo: SMD before=-2.340, after=-1.640
Pitches: SMD before=-0.923, after=-0.821
Velocity: SMD before=-0.045, after=-0.158


## 8. 결과 파일과 시각화 생성

In [9]:
effect_estimates = [
    primary_effect,
    complete_case_effect,
    name_matched_effect,
    id_matched_effect,
]
write_dict_rows(OUTPUT_DIR / "effect_estimates.csv", effect_estimates)

identification_checklist = [
    {
        "requirement": "Contemporaneous untreated comparison group",
        "status": "Not met",
        "reason": "All 2023 MLB pitchers were exposed to the new timer environment; no same-year unexposed MLB group is present.",
    },
    {
        "requirement": "Covariates measured before treatment",
        "status": "Not met",
        "reason": "Tempo, pitch count, and velocity were measured in the same season as treatment assignment.",
    },
    {
        "requirement": "Adequate positivity / overlap",
        "status": "Not met",
        "reason": f"The year-classification model had AUC {auc:.3f}; the original overlap rule removed {(~in_common_support).mean():.1%} of rows.",
    },
    {
        "requirement": "Outcome definition and source documented",
        "status": "Not met",
        "reason": "The supplied injury labels do not include the source list, timing rule, or injury inclusion criteria.",
    },
    {
        "requirement": "Consistent player inclusion criteria across years",
        "status": "Not met",
        "reason": "The export settings and minimum-pitch thresholds are undocumented; the raw tempo samples contain 391 players in 2022 and 250 unique players in 2023.",
    },
    {
        "requirement": "Repeated pre- and post-policy periods",
        "status": "Not met",
        "reason": "Only one pre-policy year and one post-policy year are available.",
    },
    {
        "requirement": "Stable measurement of exposure mechanism",
        "status": "Partly met",
        "reason": "Baseball Savant pitch tempo is release-to-release time and is not the same clock interval as the MLB pitch timer.",
    },
]
write_dict_rows(OUTPUT_DIR / "causal_identification_checklist.csv", identification_checklist)

# Figure 1: injury rates with Wilson intervals for the primary unique-player sample.
years = [2022, 2023]
risks = [primary_effect["risk_2022"], primary_effect["risk_2023"]]
counts = [primary_effect["injured_2022"], primary_effect["injured_2023"]]
totals = [primary_effect["n_2022"], primary_effect["n_2023"]]
intervals = [wilson_interval(e, n) for e, n in zip(counts, totals)]
yerr = np.array([
    [risk - low for risk, (low, high) in zip(risks, intervals)],
    [high - risk for risk, (low, high) in zip(risks, intervals)],
])
plt.figure(figsize=(7, 5))
plt.bar([str(year) for year in years], np.array(risks) * 100, yerr=yerr * 100, capsize=6)
plt.ylabel("Injury rate (%)")
plt.title("Pitcher injury labels in the unique tempo sample")
for position, (risk, events, total) in enumerate(zip(risks, counts, totals)):
    plt.text(position, risk * 100 + 1.5, f"{risk:.1%}\n({events}/{total})", ha="center")
plt.ylim(0, max((np.array(risks) + yerr[1]) * 100) + 8)
plt.tight_layout()
plt.savefig(FIGURE_DIR / "injury_rates_by_year.png", dpi=180)
plt.close()

# Figure 2: risk-difference forest plot across analysis samples.
labels = [row["analysis"] for row in effect_estimates]
rd = np.array([row["risk_difference"] * 100 for row in effect_estimates])
low = np.array([row["risk_difference_ci_low"] * 100 for row in effect_estimates])
high = np.array([row["risk_difference_ci_high"] * 100 for row in effect_estimates])
y_positions = np.arange(len(labels))
plt.figure(figsize=(10, 5.5))
plt.errorbar(rd, y_positions, xerr=[rd - low, high - rd], fmt="o", capsize=5)
plt.axvline(0, linewidth=1)
plt.yticks(y_positions, labels)
plt.xlabel("2023 minus 2022 injury-rate difference (percentage points)")
plt.title("Sensitivity of the before-after estimate to sample definition")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "effect_estimates_forest.png", dpi=180)
plt.close()

# Figure 3: paired injury-transition matrix for the 188-player name-matched panel.
transition = np.array([
    [name_matched_effect["transition_00"], name_matched_effect["transition_01"]],
    [name_matched_effect["transition_10"], name_matched_effect["transition_11"]],
])
plt.figure(figsize=(6, 5))
plt.imshow(transition)
plt.xticks([0, 1], ["No injury", "Injury"])
plt.yticks([0, 1], ["No injury", "Injury"])
plt.xlabel("2023")
plt.ylabel("2022")
plt.title("Injury-label transitions among 188 matched pitchers")
for row_index in range(2):
    for column_index in range(2):
        plt.text(column_index, row_index, str(transition[row_index, column_index]), ha="center", va="center")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "matched_injury_transitions.png", dpi=180)
plt.close()

# Figure 4: propensity-score overlap from the original post-treatment model.
plt.figure(figsize=(9, 5))
plt.hist(propensity_score[y == 0], bins=30, alpha=0.6, label="2022")
plt.hist(propensity_score[y == 1], bins=30, alpha=0.6, label="2023")
plt.xlabel("Predicted probability of being a 2023 observation")
plt.ylabel("Rows")
plt.title("Original propensity model: limited overlap between years")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "propensity_score_overlap.png", dpi=180)
plt.close()

# Figure 5: standardized mean differences before and after trimming.
variables = [row["variable"] for row in balance_rows]
before_values = np.array([row["smd_before_trimming"] for row in balance_rows])
after_values = np.array([row["smd_after_trimming"] for row in balance_rows])
y_positions = np.arange(len(variables))
plt.figure(figsize=(8, 5))
for index in range(len(variables)):
    plt.plot([before_values[index], after_values[index]], [index, index], marker="o")
plt.axvline(0, linewidth=1)
plt.axvline(0.1, linewidth=1, linestyle="--")
plt.axvline(-0.1, linewidth=1, linestyle="--")
plt.yticks(y_positions, variables)
plt.xlabel("Standardized mean difference")
plt.title("Trimming did not balance the original same-season covariates")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "covariate_balance.png", dpi=180)
plt.close()

# Figure 6: sample-flow counts by year.
stages = ["Raw tempo rows", "Unique players", "Velocity-merged", "Matched by name"]
counts_2022 = [len(raw_2022_tempo), len(tempo_2022), len(complete_2022), len(common_name_keys)]
counts_2023 = [len(raw_2023_tempo), len(tempo_2023), len(complete_2023), len(common_name_keys)]
x = np.arange(len(stages))
width = 0.36
plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, counts_2022, width, label="2022")
plt.bar(x + width / 2, counts_2023, width, label="2023")
plt.xticks(x, stages, rotation=15)
plt.ylabel("Rows or unique pitchers")
plt.title("Analysis sample flow")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "sample_flow.png", dpi=180)
plt.close()

# Figure 7: matched pitch-tempo change (descriptive mechanism check).
tempo_before = np.array([row["tempo_2022"] for row in matched_name_panel], dtype=float)
tempo_after = np.array([row["tempo_2023"] for row in matched_name_panel], dtype=float)
plt.figure(figsize=(6, 6))
plt.scatter(tempo_before, tempo_after, alpha=0.65)
minimum = min(tempo_before.min(), tempo_after.min())
maximum = max(tempo_before.max(), tempo_after.max())
plt.plot([minimum, maximum], [minimum, maximum], linewidth=1)
plt.xlabel("2022 pitch tempo (seconds, bases empty)")
plt.ylabel("2023 pitch tempo (seconds, bases empty)")
plt.title("Pitch-tempo change among matched pitchers")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "matched_tempo_change.png", dpi=180)
plt.close()

print("Generated output CSV files and seven figures.")

Generated output CSV files and seven figures.


## 9. 결론

- 중복을 제거한 전체 템포 표본에서 2023년 부상률은 2022년보다 높았지만, 위험도 차이의 95% 신뢰구간은 0을 포함했습니다.
- 동일 투수 188명을 비교한 결과도 신뢰구간이 넓고 McNemar 검정에서 뚜렷한 전후 차이를 확인하지 못했습니다.
- 분석 표본을 전체 템포 자료, 구속이 결합된 완전 사례, 이름 매칭 패널, Statcast ID 패널로 바꾸면 위험도 차이의 방향과 크기가 달라졌습니다.
- 연도별 원본 표본은 391명과 250명으로 차이가 크지만, 데이터 추출 설정과 최소 투구 수 기준이 제공되지 않아 모집단의 일관성을 확인하지 못했습니다.
- 기존 성향점수 모형은 같은 시즌에 측정된 템포·투구 수·구속을 사용했고, 연도를 거의 완벽하게 구분했으며, 공통지지 구간 적용 후에도 공변량 불균형이 크게 남았습니다.

따라서 이 데이터로 `피치클락이 부상 확률을 20.8% 높였다`는 인과 결론을 유지하지 않습니다. 이 프로젝트의 최종 산출물은 **전후 연관성 추정, 표본 민감도 분석, 인과 식별 조건 진단**입니다.

### 인과효과를 더 설득력 있게 추정하려면

1. 여러 시즌의 도입 전·후 자료를 확보합니다.
2. 명확한 부상 정의, 부상 발생일, 복귀일, 원천 URL을 기록합니다.
3. 처치 이전에 측정된 나이, 과거 부상, 과거 이닝·투구 수, 구속, 보직을 사용합니다.
4. 같은 시기의 비노출 비교군 또는 적절한 대체 통제집단을 설계합니다.
5. 인터럽티드 시계열, 차분의 차분, 이벤트 스터디와 사전 추세 검정을 검토합니다.